# ROI spatial plots and cell-type composition (10x Xenium)

For selected regions of interest (ROIs) in Xenium heart sections, this notebook produces:

1. **ROI morphology zoom-ins** — annotated cell boundaries drawn on the Xenium DAPI
   morphology image, with a scale bar.
2. **ROI-on-section overviews** — whole-section spatial scatter plots of ventricular
   cardiomyocyte (VCM) populations with the ROI drawn as a rectangle.
3. **ROI cell-type composition** — stacked bar charts and pie charts of the cell-type
   composition inside each ROI.

Sections are grouped by condition (BE vs. WT/PBS) and by ROI sizing convention
(same-sized 100,000 µm² ROIs vs. the original per-sample ROI sizes).

## Inputs

| Input | Default location | Description |
| --- | --- | --- |
| Annotated AnnData | `data/combined_Xenium_..._newcolours.h5ad` | Clustered, annotated, colour-assigned object with `obsm["spatial"]` in µm and cell types in `obs["celltype4"]` |
| SpatialData stores | `data/zarr/<sample>_with_morphology.zarr` | Per-sample Xenium objects with `images["morphology_focus"]` and `shapes["cell_boundaries"]` |

All paths are set in the **Configuration** cell below and can be overridden with
environment variables, so no absolute or machine-specific paths appear anywhere in
this notebook. The raw data is not distributed with this repository.

## Outputs

Vector PDFs written under `results/figures/` (override with `XENIUM_FIG_DIR`):

| Subdirectory | Contents |
| --- | --- |
| `roi_zoomin_same_sized/` | BE ROI morphology zoom-ins + BE ROI composition |
| `roi_on_scatter_same_sized/` | WT/PBS and BE ROI-on-section overviews (100,000 µm² ROIs) + WT/PBS composition |
| `roi_on_scatter_original_sizes/` | ROI-on-section overviews using the original per-sample ROI sizes |

## How to run

Run top to bottom. Sections 5–8 are independent of each other and can be run
selectively, but all of them depend on sections 1–4.

## 1. Setup

In [ ]:
import logging
import os
from pathlib import Path

import matplotlib as mpl
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import spatialdata as sd
import spatialdata_plot  # noqa: F401  (registers the .pl accessor on SpatialData)
from matplotlib.font_manager import FontProperties
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar

### Configuration

Paths default to a `data/` folder next to the notebook. Point them somewhere else by
exporting the environment variables below before launching Jupyter, e.g.

```bash
export XENIUM_DATA_DIR=/path/to/your/data
export XENIUM_FIG_DIR=/path/to/your/figures
```

In [ ]:
# --- Input paths -----------------------------------------------------------
DATA_DIR = Path(os.environ.get("XENIUM_DATA_DIR", "data"))

ADATA_FILENAME = (
    "combined_Xenium_largecells_noWT3_afterclustering_withleiden_annotated_"
    "ordered_withTcells_ReannotatedIfgga4_newcolours.h5ad"
)
ADATA_PATH = Path(os.environ.get("XENIUM_ADATA_PATH", DATA_DIR / ADATA_FILENAME))

# Per-sample SpatialData stores, named "<sample>_with_morphology.zarr".
ZARR_DIR = Path(os.environ.get("XENIUM_ZARR_DIR", DATA_DIR / "zarr"))

# --- Output paths ----------------------------------------------------------
FIG_ROOT = Path(os.environ.get("XENIUM_FIG_DIR", "results/figures"))

FIG_DIRS = {
    "roi_zoomin_same_sized": FIG_ROOT / "roi_zoomin_same_sized",
    "roi_on_scatter_same_sized": FIG_ROOT / "roi_on_scatter_same_sized",
    "roi_on_scatter_original_sizes": FIG_ROOT / "roi_on_scatter_original_sizes",
}

for path in FIG_DIRS.values():
    path.mkdir(parents=True, exist_ok=True)


def zarr_path(sample: str) -> Path:
    """Path to the SpatialData store for one sample."""
    return ZARR_DIR / f"{sample}_with_morphology.zarr"


print("AnnData :", ADATA_PATH)
print("Zarr dir:", ZARR_DIR)
print("Figures :", FIG_ROOT.resolve())

In [ ]:
# TrueType fonts so text stays editable in Illustrator/Inkscape.
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42

# fontTools is very chatty when subsetting fonts for PDF export.
logging.getLogger("fontTools").setLevel(logging.WARNING)
logging.getLogger("fontTools.subset").setLevel(logging.WARNING)

sc.settings.set_figure_params(dpi_save=300, transparent=True)

## 2. Shared constants

Cell-type order, colours and the highlight sets are defined once here so that the
spatial plots, the stacked bars and the pie charts all stay consistent. The order of
`CELLTYPE_COLORS` also fixes the legend order and the polygon drawing order.

In [ ]:
# 10x Xenium morphology images: typically 0.2125 µm per pixel.
UM_PER_PX = 0.2125
PX_PER_UM = 1 / UM_PER_PX

# Column in adata.obs holding the final cell-type annotation.
CELLTYPE_COL = "celltype4"

CELLTYPE_ORDER = [
    "Basal VCMs",
    "IFN-associated VCMs",
    "Stressed VCMs",
    "Remodelled VCMs",
    "Myeloid",
    "Lymphoid",
    "Other",
]

CELLTYPE_COLORS = {
    "Basal VCMs": "#2C7BB6",           # blue
    "IFN-associated VCMs": "#FFD92F",  # gold
    "Stressed VCMs": "#D7191C",        # red
    "Remodelled VCMs": "#1A9641",      # green
    "Myeloid": "#FF7A00",              # orange
    "Lymphoid": "#7B2CFF",             # purple
    "Other": "lightgrey",
}

# Cell types kept as-is in the ROI plots; everything else collapses to "Other".
HIGHLIGHT_CELLTYPES = [ct for ct in CELLTYPE_ORDER if ct != "Other"]

# On the whole-section scatter plots: which populations are coloured at all, and
# which are drawn last so they are not hidden underneath the others.
SCATTER_CELLTYPES = [
    "IFN-associated VCMs",
    "Stressed VCMs",
    "Basal VCMs",
    "Remodelled VCMs",
    "Lymphoid",
    "Myeloid",
]
SCATTER_DRAW_LAST = [
    "IFN-associated VCMs",
    "Lymphoid",
    "Myeloid",
]

# Fixed axis limits keep 1000 µm the same physical length in every exported figure.
SCATTER_XLIM = (-500, 5500)
SCATTER_YLIM_BE = (5500, -500)
SCATTER_YLIM_WT_PBS = (6000, -500)

## 3. Helper functions

The original per-sample code blocks were near-identical, so they are collected here
as functions and driven by small per-sample configuration dictionaries in sections
5–8.

In [ ]:
def build_roi_squares(roi_centers: dict) -> pd.DataFrame:
    """Turn ROI centres and target areas into square ROI bounds.

    Parameters
    ----------
    roi_centers
        Mapping ``sample -> (x_center_um, y_center_um, roi_area_um2)``.

    Returns
    -------
    DataFrame indexed by sample, with the ROI centre, bounds, side length and area
    in µm.
    """
    roi_squares = {}

    for sample, (x_center, y_center, roi_area_um2) in roi_centers.items():
        side_um = np.sqrt(roi_area_um2)
        half = side_um / 2

        roi_squares[sample] = {
            "x_center": x_center,
            "y_center": y_center,
            "x_min": x_center - half,
            "x_max": x_center + half,
            "y_min": y_center - half,
            "y_max": y_center + half,
            "side_um": side_um,
            "area_um2": roi_area_um2,
        }

    return pd.DataFrame.from_dict(roi_squares, orient="index")


def assign_rois(adata, roi_squares_df: pd.DataFrame) -> None:
    """Flag cells falling inside their sample's ROI, in place.

    Adds/overwrites ``obs["x"]``, ``obs["y"]``, ``obs["in_roi"]``, ``obs["roi_id"]``
    and ``obs["roi_area_um2"]``. Existing values are reset first, so the same AnnData
    object can be reused across sections with different ROI definitions.
    """
    adata.obs["x"] = adata.obsm["spatial"][:, 0]
    adata.obs["y"] = adata.obsm["spatial"][:, 1]

    adata.obs["in_roi"] = False
    # Object dtype, not float: roi_id holds sample-name strings. Initialising it with
    # a bare np.nan makes it float64, and writing strings into it raises TypeError on
    # pandas >= 2.2.
    adata.obs["roi_id"] = np.full(adata.n_obs, np.nan, dtype=object)
    adata.obs["roi_area_um2"] = np.nan

    for sample, roi in roi_squares_df.iterrows():
        mask = (
            (adata.obs["name"] == sample)
            & (adata.obs["x"] >= roi["x_min"])
            & (adata.obs["x"] <= roi["x_max"])
            & (adata.obs["y"] >= roi["y_min"])
            & (adata.obs["y"] <= roi["y_max"])
        )

        adata.obs.loc[mask, "in_roi"] = True
        adata.obs.loc[mask, "roi_id"] = sample
        adata.obs.loc[mask, "roi_area_um2"] = roi["area_um2"]


def roi_bounds_px(roi_squares_df: pd.DataFrame, sample: str) -> tuple:
    """ROI bounds for one sample, converted from µm to morphology-image pixels."""
    roi = roi_squares_df.loc[sample]

    return (
        roi["x_min"] * PX_PER_UM,
        roi["x_max"] * PX_PER_UM,
        roi["y_min"] * PX_PER_UM,
        roi["y_max"] * PX_PER_UM,
    )

In [ ]:
def load_roi_cell_shapes(sdata, adata, sample: str):
    """Cell boundary polygons for one sample, annotated and clipped to its ROI.

    Cell types outside ``HIGHLIGHT_CELLTYPES`` are collapsed into "Other" so the
    overlays match the composition plots. Polygons are returned in morphology-image
    pixel coordinates, which is what :func:`plot_roi_dapi` expects.
    """
    cell_shapes = sdata.shapes["cell_boundaries"].copy()

    # Drop annotation columns left over from any previous merge.
    cols_to_drop = [
        "cell_id", CELLTYPE_COL, "in_roi",
        "celltype_x", "celltype_y",
        "in_roi_x", "in_roi_y",
        "plot_color", "celltype_plot",
    ]
    cell_shapes = cell_shapes.drop(
        columns=[c for c in cols_to_drop if c in cell_shapes.columns],
        errors="ignore",
    )

    anno = (
        adata.obs
        .loc[adata.obs["name"] == sample, ["cell_id", CELLTYPE_COL, "in_roi"]]
        .copy()
    )

    cell_shapes_anno = cell_shapes.merge(
        anno,
        left_index=True,
        right_on="cell_id",
        how="left",
    )

    shapes_roi = cell_shapes_anno[cell_shapes_anno["in_roi"].fillna(False)].copy()

    celltypes = shapes_roi[CELLTYPE_COL].astype(str)
    shapes_roi["celltype_plot"] = celltypes.where(
        celltypes.isin(HIGHLIGHT_CELLTYPES),
        "Other",
    )
    shapes_roi["plot_color"] = shapes_roi["celltype_plot"].map(CELLTYPE_COLORS)

    # µm -> morphology-image pixels
    shapes_roi["geometry"] = shapes_roi.geometry.scale(
        xfact=PX_PER_UM,
        yfact=PX_PER_UM,
        origin=(0, 0),
    )

    return shapes_roi

In [ ]:
def plot_roi_dapi(
    sdata,
    cell_shapes_roi_px,
    sample,
    out_dir,
    x0_px, x1_px, y0_px, y1_px,
    celltype_colors=None,
    scale="scale0",
    channel="DAPI",
    overlay_mode="boundary",   # "boundary", "filled", or "none"
    nuclei_display="gray",     # "gray" or "blue"
    fill_alpha=0.5,
    boundary_linewidth=0.4,
    filled_edgecolor="#777777",
    filled_linewidth=0.15,
    show_legend=True,
    figsize=(6, 6),
    dpi=300,
    # scale bar
    show_scale_bar=True,
    scale_bar_um=50,
    um_per_coord=UM_PER_PX,
    scale_bar_color="white",
    scale_bar_linewidth=3,
    scale_bar_fontsize=8,
    scale_bar_location="lower right",
    save_label=None,
):
    """Plot one ROI of the DAPI morphology image with cell-type overlays.

    ``x0_px``/``x1_px``/``y0_px``/``y1_px`` and ``cell_shapes_roi_px`` must all be in
    scale0 pixel coordinates, matching the x/y coordinates of the SpatialData image.

    Returns
    -------
    (fig, ax, out_file)
    """
    if celltype_colors is None:
        celltype_colors = CELLTYPE_COLORS

    img_xr = sdata.images["morphology_focus"][scale]["image"]

    cell_shapes_roi_px = cell_shapes_roi_px.copy()
    cell_shapes_roi_px["plot_color"] = (
        cell_shapes_roi_px["celltype_plot"].map(celltype_colors)
    )

    img_crop = img_xr.sel(
        c=channel,
        x=slice(x0_px, x1_px),
        y=slice(y0_px, y1_px),
    ).data.compute()

    # Percentile-based contrast stretch.
    vmin, vmax = np.percentile(img_crop, [2, 99.5])
    img_norm = np.clip((img_crop - vmin) / (vmax - vmin + 1e-8), 0, 1)

    fig, ax = plt.subplots(figsize=figsize)

    if nuclei_display == "gray":
        ax.imshow(
            img_crop,
            cmap="gray",
            vmin=vmin,
            vmax=vmax,
            extent=[x0_px, x1_px, y1_px, y0_px],
        )
    elif nuclei_display == "blue":
        rgb = np.zeros((*img_norm.shape, 3), dtype=float)
        rgb[..., 2] = img_norm
        ax.imshow(rgb, extent=[x0_px, x1_px, y1_px, y0_px])
    else:
        raise ValueError("nuclei_display must be 'gray' or 'blue'")

    # Draw in the order of celltype_colors so legend and z-order stay consistent.
    present_celltypes = [
        ct for ct in celltype_colors
        if ct in cell_shapes_roi_px["celltype_plot"].values
    ]

    if overlay_mode == "boundary":
        for ct in present_celltypes:
            sub = cell_shapes_roi_px[cell_shapes_roi_px["celltype_plot"] == ct]
            if len(sub) > 0:
                sub.boundary.plot(
                    ax=ax,
                    edgecolor=celltype_colors[ct],
                    linewidth=boundary_linewidth,
                )
    elif overlay_mode == "filled":
        for ct in present_celltypes:
            sub = cell_shapes_roi_px[cell_shapes_roi_px["celltype_plot"] == ct]
            if len(sub) > 0:
                sub.plot(
                    ax=ax,
                    color=celltype_colors[ct],
                    edgecolor=filled_edgecolor,
                    linewidth=filled_linewidth,
                    alpha=fill_alpha,
                )
    elif overlay_mode == "none":
        pass
    else:
        raise ValueError("overlay_mode must be 'boundary', 'filled', or 'none'")

    ax.set_xlim(x0_px, x1_px)
    ax.set_ylim(y1_px, y0_px)
    ax.set_aspect("equal")
    ax.set_title(f"{sample}: {channel}", fontsize=12)
    ax.set_xticks([])
    ax.set_yticks([])

    if show_scale_bar:
        _add_manual_scale_bar(
            ax,
            x0_px, x1_px, y0_px, y1_px,
            scale_bar_um=scale_bar_um,
            um_per_coord=um_per_coord,
            color=scale_bar_color,
            linewidth=scale_bar_linewidth,
            fontsize=scale_bar_fontsize,
            location=scale_bar_location,
        )

    if show_legend and overlay_mode != "none":
        if overlay_mode == "boundary":
            handles = [
                Line2D([0], [0], color=celltype_colors[ct], lw=2, label=ct)
                for ct in present_celltypes
            ]
        else:
            handles = [
                mpatches.Patch(color=celltype_colors[ct], label=ct)
                for ct in present_celltypes
            ]

        ax.legend(
            handles=handles,
            title="Cell type",
            bbox_to_anchor=(1.02, 1),
            loc="upper left",
            borderaxespad=0,
        )

    plt.tight_layout()

    if save_label is None:
        save_label = f"ROI_DAPI_{overlay_mode}_{nuclei_display}"

    out_file = Path(out_dir) / f"{sample}_{save_label}.pdf"
    fig.savefig(out_file, dpi=dpi, bbox_inches="tight", transparent=True)
    print(f"Saved: {out_file}")

    return fig, ax, out_file


def _add_manual_scale_bar(
    ax,
    x0_px, x1_px, y0_px, y1_px,
    scale_bar_um,
    um_per_coord,
    color,
    linewidth,
    fontsize,
    location,
):
    """Draw a scale bar in data (pixel) coordinates, inset from the given corner."""
    scale_bar_px = scale_bar_um / um_per_coord

    x_range = x1_px - x0_px
    y_range = y1_px - y0_px
    pad_x = 0.05 * x_range
    pad_y = 0.05 * y_range

    if location not in {"lower right", "lower left", "upper right", "upper left"}:
        raise ValueError(
            "scale_bar_location must be 'lower right', 'lower left', "
            "'upper right', or 'upper left'"
        )

    vertical, horizontal = location.split()

    if horizontal == "right":
        x_end = x1_px - pad_x
        x_start = x_end - scale_bar_px
    else:
        x_start = x0_px + pad_x
        x_end = x_start + scale_bar_px

    # The y axis is inverted (image convention), hence "lower" uses y1_px.
    if vertical == "lower":
        y = y1_px - pad_y
        text_y = y - 0.025 * y_range
        va = "bottom"
    else:
        y = y0_px + pad_y
        text_y = y + 0.025 * y_range
        va = "top"

    ax.plot(
        [x_start, x_end],
        [y, y],
        color=color,
        linewidth=linewidth,
        solid_capstyle="butt",
    )
    ax.text(
        (x_start + x_end) / 2,
        text_y,
        f"{scale_bar_um} µm",
        color=color,
        ha="center",
        va=va,
        fontsize=fontsize,
    )

In [ ]:
def plot_section_with_roi(
    adata,
    sample: str,
    roi_squares_df: pd.DataFrame,
    out_dir,
    ylim,
    xlim=SCATTER_XLIM,
    figsize=(8, 8),
    spot_size=25,
    scalebar_um=1000,
    dpi=600,
    filename=None,
):
    """Whole-section spatial scatter of the selected populations, with the ROI boxed.

    Cell types outside ``SCATTER_CELLTYPES`` are set to NaN so they render in the
    ``na_color`` grey. Points are drawn grey first, then the remaining selected
    populations, then ``SCATTER_DRAW_LAST`` — reordering rows rather than replotting,
    so every point keeps the same size.
    """
    sample_adata = adata[adata.obs["name"] == sample].copy()

    celltypes = sample_adata.obs[CELLTYPE_COL].astype(str)
    sample_adata.obs.loc[~celltypes.isin(SCATTER_CELLTYPES), CELLTYPE_COL] = np.nan

    draw_rank = np.select(
        [
            sample_adata.obs[CELLTYPE_COL].isna(),
            sample_adata.obs[CELLTYPE_COL].astype(str).isin(SCATTER_DRAW_LAST),
        ],
        [0, 2],
        default=1,
    )
    sample_adata = sample_adata[np.argsort(draw_rank)].copy()

    fig, ax = plt.subplots(figsize=figsize)

    sc.pl.spatial(
        sample_adata,
        color=CELLTYPE_COL,
        na_color="lightgrey",
        na_in_legend=False,
        spot_size=spot_size,
        ax=ax,
        show=False,
    )

    # ROI rectangle. Both the ROI table and obsm["spatial"] are in µm.
    roi = roi_squares_df.loc[sample]
    x0, x1 = roi["x_min"], roi["x_max"]
    y0, y1 = roi["y_min"], roi["y_max"]
    print(f"{sample} ROI rectangle: x {x0}-{x1}, y {y0}-{y1}")

    ax.add_patch(
        Rectangle(
            (x0, y0),
            x1 - x0,
            y1 - y0,
            linewidth=2,
            edgecolor="black",
            facecolor="none",
            zorder=20,
        )
    )

    # Fixed limits + equal aspect keep the physical scale identical across samples.
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)
    ax.set_aspect("equal", adjustable="box")

    ax.add_artist(
        AnchoredSizeBar(
            ax.transData,
            scalebar_um,
            f"{scalebar_um} µm",
            loc="lower right",
            pad=0.3,
            borderpad=0.5,
            sep=4,
            color="black",
            frameon=False,
            size_vertical=20,
            fontproperties=FontProperties(size=10),
        )
    )

    ax.set_title(f"{sample}: VCM populations with ROI")
    ax.set_xticks([])
    ax.set_yticks([])

    # Fixed margins keep the exported canvas size consistent between samples.
    fig.subplots_adjust(left=0.05, right=0.82, bottom=0.05, top=0.92)

    if filename is None:
        filename = f"{sample}_VCMs_highlighted_ROI_scalebar_fixed_scale.pdf"

    out_file = Path(out_dir) / filename
    fig.savefig(out_file, dpi=dpi)
    print(f"Saved: {out_file}")

    plt.show()
    plt.close(fig)

    return out_file

In [ ]:
def composition_table(adata_roi, sample_order) -> pd.DataFrame:
    """Percentage cell-type composition per sample, inside the ROIs.

    Cell types are collapsed to the same categories used in the spatial plots, and
    rows/columns are reindexed so that missing samples or cell types appear as zeros
    in a fixed order.
    """
    composition_df = adata_roi.obs[["name", CELLTYPE_COL]].copy()

    celltypes = composition_df[CELLTYPE_COL].astype(str)
    composition_df["celltype_plot"] = celltypes.where(
        celltypes.isin(HIGHLIGHT_CELLTYPES),
        "Other",
    )

    counts = (
        composition_df
        .groupby(["name", "celltype_plot"], observed=True)
        .size()
        .unstack(fill_value=0)
        .reindex(index=sample_order, columns=CELLTYPE_ORDER, fill_value=0)
    )

    return counts.div(counts.sum(axis=1), axis=0) * 100


def _celltype_legend(ax, celltypes=None):
    """Attach a cell-type legend in CELLTYPE_ORDER, outside the axes."""
    if celltypes is None:
        celltypes = CELLTYPE_ORDER

    handles = [
        mpatches.Patch(color=CELLTYPE_COLORS[ct], label=ct)
        for ct in celltypes
    ]
    ax.legend(
        handles=handles,
        title="Cell type",
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
        borderaxespad=0,
    )


def plot_composition_stacked(
    composition_pct: pd.DataFrame,
    out_file,
    figsize=(5, 5),
    show_values=False,
    min_label_pct=3,
):
    """One stacked bar per sample.

    Segments are stacked in reversed ``CELLTYPE_ORDER`` so that the top-to-bottom
    order of the bar matches the top-to-bottom order of the legend.
    """
    fig, ax = plt.subplots(figsize=figsize)

    x = np.arange(len(composition_pct.index))
    bottom = np.zeros(len(composition_pct))

    for ct in reversed(CELLTYPE_ORDER):
        values = composition_pct[ct].values
        ax.bar(x, values, bottom=bottom, width=0.75, color=CELLTYPE_COLORS[ct])

        if show_values:
            for i, value in enumerate(values):
                if value >= min_label_pct:
                    ax.text(
                        x[i],
                        bottom[i] + value / 2,
                        f"{value:.1f}",
                        ha="center",
                        va="center",
                        fontsize=8,
                        color="black",
                    )

        bottom += values

    ax.set_ylabel("Cell-type composition (%)")
    ax.set_xlabel("")
    ax.set_ylim(0, 100)
    ax.set_xticks(x)
    ax.set_xticklabels(composition_pct.index, rotation=45, ha="right")
    ax.grid(False)

    _celltype_legend(ax)
    plt.tight_layout()

    out_file = Path(out_file)
    fig.savefig(out_file, dpi=300, bbox_inches="tight", transparent=True)
    print(f"Saved: {out_file}")

    plt.show()
    plt.close(fig)


def plot_composition_single_bars(
    composition_pct: pd.DataFrame,
    out_dir,
    figsize=(1.2, 8),
    min_label_pct=3,
    suffix="celltype_composition",
):
    """One tall single-bar figure per sample, saved separately."""
    for sample in composition_pct.index:
        fig, ax = plt.subplots(figsize=figsize)
        bottom = 0

        for ct in reversed(CELLTYPE_ORDER):
            value = composition_pct.loc[sample, ct]
            ax.bar(0, value, bottom=bottom, width=0.65, color=CELLTYPE_COLORS[ct])

            if value >= min_label_pct:
                ax.text(
                    0,
                    bottom + value / 2,
                    f"{value:.1f}",
                    ha="center",
                    va="center",
                    fontsize=16,
                    color="black",
                )

            bottom += value

        ax.set_ylim(0, 100)
        ax.set_ylabel("Cell-type composition (%)")
        ax.set_xlabel("")
        ax.set_xticks([0])
        ax.set_xticklabels([sample])
        ax.grid(False)

        _celltype_legend(ax)
        plt.tight_layout()

        out_file = Path(out_dir) / f"{sample}_{suffix}.pdf"
        fig.savefig(out_file, dpi=300, bbox_inches="tight", transparent=True)
        print(f"Saved: {out_file}")

        plt.show()
        plt.close(fig)


def plot_composition_pies(
    composition_pct: pd.DataFrame,
    out_dir,
    figsize=(8, 8),
    min_label_pct=3,
    suffix="celltype_composition_pie",
):
    """One pie chart per sample. Cell types at 0% are omitted from wedges and legend."""
    for sample in composition_pct.index:
        values = composition_pct.loc[sample, CELLTYPE_ORDER]
        present = values > 0
        values_plot = values[present]
        celltypes_plot = list(values.index[present])

        fig, ax = plt.subplots(figsize=figsize)

        _, _, autotexts = ax.pie(
            values_plot,
            colors=[CELLTYPE_COLORS[ct] for ct in celltypes_plot],
            startangle=90,
            counterclock=False,
            autopct=lambda pct: f"{pct:.1f}%" if pct >= min_label_pct else "",
            textprops={"fontsize": 14, "color": "black"},
            wedgeprops={"edgecolor": "white", "linewidth": 1},
        )

        for autotext in autotexts:
            autotext.set_fontsize(14)

        ax.axis("equal")
        ax.set_title(f"{sample}", fontsize=16)

        _celltype_legend(ax, celltypes=celltypes_plot)
        plt.tight_layout()

        out_file = Path(out_dir) / f"{sample}_{suffix}.pdf"
        fig.savefig(out_file, dpi=300, bbox_inches="tight", transparent=True)
        print(f"Saved: {out_file}")

        plt.show()
        plt.close(fig)

## 4. Load the annotated object

The AnnData object is read once. Each section below re-assigns the ROI columns in
`obs` for its own ROI definition, so the object can be reused throughout.

In [ ]:
adata = sc.read_h5ad(ADATA_PATH)
adata

## 5. BE ROI morphology zoom-ins and composition

Same-sized 100,000 µm² ROIs for the four BE replicates. Annotated cell masks are
drawn filled on the DAPI morphology image, followed by the ROI cell-type composition
plots.

The inline comments on `BE_ROI_CENTERS` record the ROI number used in the figures and
the ROI area used in earlier, non-size-matched versions of these panels.

In [ ]:
out_dir = FIG_DIRS["roi_zoomin_same_sized"]

BE_ROI_CENTERS = {
    # sample: (x_center_um, y_center_um, roi_area_um2)
    "BE_rep1": (1032, 3520, 100_000),  # ROI2; small lymphoid cluster, imperfect segmentation. Earlier area: 50,000
    "BE_rep2": (3070, 2347, 100_000),  # ROI1. Earlier area: 300,000
    "BE_rep3": (3694, 1151, 100_000),  # ROI3 in figures; clearest example. Earlier area: 150,000
    "BE_rep4": (1814, 2032, 100_000),  # ROI3. Earlier area: 100,000
}

# Per-sample plotting overrides, in the order the figures were generated.
BE_ZOOM_SETTINGS = {
    "BE_rep3": {"save_label": "filled_DAPI_with_scalebar_darker", "boundary_linewidth": 0.1},
    "BE_rep1": {"save_label": "_ROI2_zoomedin_darker", "boundary_linewidth": 0},
    "BE_rep4": {"save_label": "ROI3_filled_DAPI_with_scalebar_darker", "boundary_linewidth": 0},
    "BE_rep2": {"save_label": "ROI1_filled_DAPI_with_scalebar_darker", "boundary_linewidth": 0},
}

BE_SAMPLE_ORDER = ["BE_rep1", "BE_rep2", "BE_rep3", "BE_rep4"]

be_roi_squares = build_roi_squares(BE_ROI_CENTERS)
be_roi_squares

In [ ]:
assign_rois(adata, be_roi_squares)
adata_roi = adata[adata.obs["in_roi"]].copy()

adata_roi.obs["name"].value_counts()

### 5.1 Morphology zoom-ins

In [ ]:
for sample, settings in BE_ZOOM_SETTINGS.items():
    sdata = sd.read_zarr(zarr_path(sample))
    shapes_px = load_roi_cell_shapes(sdata, adata, sample)
    x0_px, x1_px, y0_px, y1_px = roi_bounds_px(be_roi_squares, sample)

    print(f"{sample}: {len(shapes_px)} cells in ROI; "
          f"pixel bounds x {x0_px:.0f}-{x1_px:.0f}, y {y0_px:.0f}-{y1_px:.0f}")

    fig, ax, _ = plot_roi_dapi(
        sdata=sdata,
        cell_shapes_roi_px=shapes_px,
        sample=sample,
        out_dir=out_dir,
        x0_px=x0_px, x1_px=x1_px,
        y0_px=y0_px, y1_px=y1_px,
        scale="scale0",
        overlay_mode="filled",
        fill_alpha=0.65,
        nuclei_display="gray",
        show_scale_bar=True,
        scale_bar_um=50,
        **settings,
    )
    plt.show()
    plt.close(fig)

### 5.2 ROI cell-type composition

In [ ]:
composition_pct = composition_table(adata_roi, BE_SAMPLE_ORDER)
composition_pct

In [ ]:
plot_composition_stacked(
    composition_pct,
    out_dir / "ROI_celltype_composition.pdf",
    figsize=(5, 5),
)

plot_composition_stacked(
    composition_pct,
    out_dir / "ROI_celltype_composition_withpercentage.pdf",
    figsize=(5, 5),
    show_values=True,
)

plot_composition_single_bars(composition_pct, out_dir)
plot_composition_pies(composition_pct, out_dir)

## 6. WT/PBS ROI-on-section overviews and composition

Same-sized 100,000 µm² ROIs for the WT and PBS control replicates. The commented-out
entries are the other candidate ROIs that were evaluated but not used in the final
figures; they are kept for provenance.

In [ ]:
out_dir = FIG_DIRS["roi_on_scatter_same_sized"]

WT_PBS_ROI_CENTERS = {
    # sample: (x_center_um, y_center_um, roi_area_um2)
    "PBS_rep2": (1404, 4964, 100_000),
    "WT_rep2": (1425, 3652, 100_000),
    # Candidate ROIs considered but not used:
    # "PBS_rep1": (1681, 3271, ...),
    # "PBS_rep2": (2950, 3964, 150_000),
    # "PBS_rep3": (3261, 3327, ...),
    # "PBS_rep4": (1336, 1119, ...),
    # "WT_rep1":  (3913,  908, 150_000),
    # "WT_rep4":  (3636, 3576, 150_000),
    # "WT_rep5":  (2090, 4000, ...),
}

WT_PBS_SAMPLE_ORDER = ["WT_rep2", "PBS_rep2"]

wt_pbs_roi_squares = build_roi_squares(WT_PBS_ROI_CENTERS)
wt_pbs_roi_squares

In [ ]:
assign_rois(adata, wt_pbs_roi_squares)
adata_roi = adata[adata.obs["in_roi"]].copy()

adata_roi.obs["name"].value_counts()

### 6.1 ROI on the whole-section scatter

In [ ]:
for sample in WT_PBS_SAMPLE_ORDER:
    plot_section_with_roi(
        adata,
        sample=sample,
        roi_squares_df=wt_pbs_roi_squares,
        out_dir=out_dir,
        ylim=SCATTER_YLIM_WT_PBS,
    )

### 6.2 WT/PBS morphology zoom-ins


In [ ]:
RUN_WT_PBS_MORPHOLOGY_ZOOMS = True

if RUN_WT_PBS_MORPHOLOGY_ZOOMS:
    for sample in WT_PBS_SAMPLE_ORDER:
        sdata = sd.read_zarr(zarr_path(sample))
        shapes_px = load_roi_cell_shapes(sdata, adata, sample)
        x0_px, x1_px, y0_px, y1_px = roi_bounds_px(wt_pbs_roi_squares, sample)

        fig, ax, _ = plot_roi_dapi(
            sdata=sdata,
            cell_shapes_roi_px=shapes_px,
            sample=sample,
            out_dir=out_dir,
            x0_px=x0_px, x1_px=x1_px,
            y0_px=y0_px, y1_px=y1_px,
            scale="scale0",
            overlay_mode="filled",
            fill_alpha=0.65,
            nuclei_display="gray",
            boundary_linewidth=0,
            scale_bar_um=50,
            save_label="filled_DAPI_with_scalebar_darker",
        )
        plt.show()
        plt.close(fig)

### 6.3 ROI cell-type composition

In [ ]:
composition_pct = composition_table(adata_roi, WT_PBS_SAMPLE_ORDER)
composition_pct

In [ ]:
plot_composition_stacked(
    composition_pct,
    out_dir / "PBS_WT_ROI_celltype_composition.pdf",
    figsize=(5, 7),
)

plot_composition_stacked(
    composition_pct,
    out_dir / "PBS_WT_ROI_celltype_composition_withpercentage.pdf",
    figsize=(5, 7),
    show_values=True,
)

plot_composition_single_bars(composition_pct, out_dir)
plot_composition_pies(composition_pct, out_dir)

## 7. BE ROI-on-section overviews (same-sized ROIs)

The same 100,000 µm² BE ROIs as section 5, shown on the whole-section scatter plots.
The BE sections use a slightly smaller y range than WT/PBS.

In [ ]:
out_dir = FIG_DIRS["roi_on_scatter_same_sized"]

assign_rois(adata, be_roi_squares)

for sample in ["BE_rep3", "BE_rep1", "BE_rep2", "BE_rep4"]:
    plot_section_with_roi(
        adata,
        sample=sample,
        roi_squares_df=be_roi_squares,
        out_dir=out_dir,
        ylim=SCATTER_YLIM_BE,
    )